In [ ]:
#this script merge two different road type dataset

#NOT POSSIBLE TO RUN IN DRIVE WITHOUT RIFINDING ORIGINAL DATASET FROM THE SITES

import geopandas as gpd
import pandas as pd
DATA_DIR = "dataset_big"

# ------------------------------------------------------------
# 1. Load the two road datasets
# ------------------------------------------------------------
frst_roads = gpd.read_file(f"{DATA_DIR}/Nigeria_roads_galga.shp") #file from https://figshare.com/articles/dataset/FRST_dataset/29424107
print(f"FRST roads: {len(frst_roads)} features")


hotosm_roads = gpd.read_file(f"{DATA_DIR}/hotosm_nga_roads_lines_gpkg.gpkg")#file from Nigeria Roads (OpenStreetMap Export) https://data.humdata.org
print(f"HOTOSM roads: {len(hotosm_roads)} features")

# ------------------------------------------------------------
# 2. Check CRS - manteniamo EPSG:3857 come richiesto
# ------------------------------------------------------------
print(f"FRST CRS: {frst_roads.crs}")
print(f"HOTOSM CRS: {hotosm_roads.crs}")

# Uniformiamo a EPSG:3857 (Web Mercator)
target_crs = "EPSG:3857"

if frst_roads.crs != target_crs:
    frst_roads = frst_roads.to_crs(target_crs)
if hotosm_roads.crs != target_crs:
    hotosm_roads = hotosm_roads.to_crs(target_crs)

print(f"CRS unificato a: {target_crs}")

# ------------------------------------------------------------
# 3. Add source column to track origin
# ------------------------------------------------------------
frst_roads['source'] = 'FRST'
hotosm_roads['source'] = 'HOTOSM'

# ------------------------------------------------------------
# 4. Concatenate the two GeoDataFrames
# ------------------------------------------------------------
combined = pd.concat([frst_roads, hotosm_roads], ignore_index=True, sort=False)
print(f"Combined (before deduplication): {len(combined)} features")

# ------------------------------------------------------------
# 5. Remove duplicate geometries
# ------------------------------------------------------------
combined['geom_wkt'] = combined.geometry.apply(lambda g: g.wkt)
combined_dedup = combined.drop_duplicates(subset='geom_wkt').drop(columns=['geom_wkt'])
print(f"After removing duplicate geometries: {len(combined_dedup)} features")

# ------------------------------------------------------------
# 6. DROP THE PROBLEMATIC COLUMN 'surface' (if it exists)
# ------------------------------------------------------------
if 'surface' in combined_dedup.columns:
    combined_dedup = combined_dedup.drop(columns=['surface'])
    print("Colonna 'surface' rimossa con successo")


# ------------------------------------------------------------
# 7. Save to a new GeoPackage
# ------------------------------------------------------------
output_file = f"{DATA_DIR}/nigeria_roads_merged.gpkg"
combined_dedup.to_file(output_file, layer='roads', driver='GPKG')
print(f"Saved merged roads to {output_file}")
print(f"CRS del file finale: {combined_dedup.crs}")

FRST roads: 1481859 features
HOTOSM roads: 1734838 features
FRST CRS: EPSG:3857
HOTOSM CRS: EPSG:4326
CRS unificato a: EPSG:3857
Combined (before deduplication): 3216697 features
After removing duplicate geometries: 3215644 features
Colonna 'surface' rimossa con successo
Saved merged roads to nigeria_roads_merged.gpkg
CRS del file finale: EPSG:3857
